# Colab 00: Generate Step Dataset (Inline, Full)
- `generate_jssp_step_dataset.py` 대응 노트북
- one-shot `train.json` -> step-by-step `jsonl` 생성
- strict/strict_makespan/fail_fast/progress_every 옵션 포함
- 필요 시 생성 결과를 HF dataset repo에 업로드


In [ ]:
!pip -q install -U huggingface_hub datasets


## 1) 설정


In [ ]:
import json
import re
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

from huggingface_hub import login, hf_hub_download, HfApi, upload_file

import os
######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

ROLE_DEFAULTS = {
    'serial': {
        'policy': {
            'output_jsonl_path': '/content/jssp_step_train_policy.jsonl',
            'output_summary_path': '/content/jssp_step_train_policy.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_policy.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_policy.summary.json',
        },
        'reason': {
            'output_jsonl_path': '/content/jssp_step_train_reason.jsonl',
            'output_summary_path': '/content/jssp_step_train_reason.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_reason_step_train_all_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_reason.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_reason.summary.json',
        },
        'both': {
            'output_jsonl_path': '/content/jssp_step_train_with_reason.jsonl',
            'output_summary_path': '/content/jssp_step_train_with_reason.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_mixed_step_train_all_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_with_reason.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_with_reason.summary.json',
        },
    },
    'dispatch': {
        'policy': {
            'output_jsonl_path': '/content/jssp_step_train_policy_dispatch.jsonl',
            'output_summary_path': '/content/jssp_step_train_policy_dispatch.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_policy_dispatch.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_policy_dispatch.summary.json',
        },
        'reason': {
            'output_jsonl_path': '/content/jssp_step_train_reason_dispatch.jsonl',
            'output_summary_path': '/content/jssp_step_train_reason_dispatch.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_reason_step_train_dispatch_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_reason_dispatch.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_reason_dispatch.summary.json',
        },
        'both': {
            'output_jsonl_path': '/content/jssp_step_train_with_reason_dispatch.jsonl',
            'output_summary_path': '/content/jssp_step_train_with_reason_dispatch.summary.json',
            'output_hf_repo': 'HYUNJINI/jssp_mixed_step_train_dispatch_v1',
            'output_hf_path_in_repo': 'train_data/jssp_step_train_with_reason_dispatch.jsonl',
            'summary_hf_path_in_repo': 'train_data/jssp_step_train_with_reason_dispatch.summary.json',
        },
    },
}

CFG = {
    # 입력 소스
    # - 'hf': HF dataset repo/file 에서 다운로드
    # - 'local': /content/... 경로에서 직접 읽기
    'input_source': 'hf',
    # argparse --input alias (직접 경로를 지정하면 source 분기보다 우선)
    'input': None,
    'input_hf_repo': 'HYUNJINI/jssp_raw_source_v1',
    'input_hf_file': 'llm_jssp/train.json',
    'input_local_path': '/content/train.json',

    # environment mode
    'env_mode': 'serial',  # serial | dispatch

    # 출력 역할
    'dataset_role': 'policy',  # policy | reason | both
    'reason_topk': 3,
    'output_jsonl_path': None,
    'output_summary_path': None,

    # notebook inline 생성 옵션 호환
    'start_idx': 0,
    'end_idx': None,
    'max_instances': None,
    'no_strict': False,
    'strict_makespan': False,
    'fail_fast': False,
    'progress_every': 500,
    'action_code_seed': 42,
    'action_code_width': 4,
    'action_code_cap': 9999,

    # 업로드 옵션
    'upload_output_to_hf': False,
    'output_hf_repo': None,
    'output_hf_path_in_repo': None,
    'upload_summary_to_hf': True,
    'summary_hf_path_in_repo': None,
}

role = str(CFG['dataset_role']).lower()
env_mode = str(CFG.get('env_mode', 'serial')).lower()
if env_mode not in ROLE_DEFAULTS:
    raise ValueError(f"Unsupported CFG['env_mode']={env_mode}")
if role not in ROLE_DEFAULTS[env_mode]:
    raise ValueError(f"Unsupported CFG['dataset_role']={role}")
for k, v in ROLE_DEFAULTS[env_mode][role].items():
    if CFG.get(k) in (None, ''):
        CFG[k] = v

print(CFG)


## 2) 유틸 함수 (inline)


In [ ]:
import json

import random

import re

from functools import lru_cache

from dataclasses import dataclass

from pathlib import Path

from typing import Dict, Iterable, List, Optional, Sequence, Tuple

ACTION_TOKEN_PREFIX = "A"

LEGACY_ACTION_TOKEN_PREFIX = "S"

ACTION_CODE_PATTERN = re.compile(
    r"(?:Action\s*:\s*)?<\s*[aAsS]\s*(\d+)\s*>",
    re.IGNORECASE,
)

def format_action_code(
    code_index: int,
    code_width: int = 4,
    prefix: str = ACTION_TOKEN_PREFIX,
) -> str:
    if int(code_index) < 0:
        raise ValueError(f"code_index must be non-negative, got {code_index}.")
    if int(code_width) < 1:
        raise ValueError(f"code_width must be >= 1, got {code_width}.")
    return f"<{str(prefix)}{int(code_index):0{int(code_width)}d}>"

def parse_action_code(text: str, code_width: int = 4) -> Optional[str]:
    if not isinstance(text, str):
        return None
    match = ACTION_CODE_PATTERN.search(text)
    if not match:
        return None
    return format_action_code(int(match.group(1)), code_width=code_width)

HEADER_PATTERN = re.compile(
    r"JSSP\s+with\s+(\d+)\s+Jobs,\s*(\d+)\s+Machines",
    re.IGNORECASE,
)

JOB_HEADER_PATTERN = re.compile(
    r"^\s*Job\s+(\d+)\s+consists\s+of\s+Operations:\s*$",
    re.IGNORECASE,
)

OPERATION_PATTERN = re.compile(
    r"^\s*Operation\s+(\d+):\s*M(\d+),\s*(\d+)\s*$",
    re.IGNORECASE,
)

SOLUTION_OPERATION_PATTERN = re.compile(
    r"Job\s*(\d+)\s*Operation\s*(\d+),\s*M(\d+)",
    re.IGNORECASE,
)

MAKESPAN_PATTERN = re.compile(r"Makespan\s*:\s*(\d+)", re.IGNORECASE)

@dataclass(frozen=True)
class ParsedTeacherAction:
    """Parsed action from one-shot output text."""

    job_id: int
    op_idx: int
    machine_id: int

def parse_prompt_jobs_first(prompt_text: str, strict: bool = True) -> List[List[List[int]]]:
    """
    Parse `prompt_jobs_first` text into `inst_for_ortools` format.

    Returns:
        inst_for_ortools[job][op] = [machine, duration]
    """
    if not isinstance(prompt_text, str) or not prompt_text.strip():
        raise ValueError("prompt_text must be a non-empty string.")

    header_match = HEADER_PATTERN.search(prompt_text)
    expected_jobs = int(header_match.group(1)) if header_match else None
    expected_machines = int(header_match.group(2)) if header_match else None

    jobs: Dict[int, List[List[int]]] = {}
    current_job: Optional[int] = None

    for raw_line in prompt_text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        job_match = JOB_HEADER_PATTERN.match(line)
        if job_match:
            current_job = int(job_match.group(1))
            jobs.setdefault(current_job, [])
            continue

        op_match = OPERATION_PATTERN.match(line)
        if op_match:
            if current_job is None:
                raise ValueError(f"Operation line appears before job header: {raw_line!r}")

            op_idx = int(op_match.group(1))
            machine = int(op_match.group(2))
            duration = int(op_match.group(3))
            expected_op_idx = len(jobs[current_job])
            if strict and op_idx != expected_op_idx:
                raise ValueError(
                    f"Operation index mismatch in Job {current_job}: "
                    f"expected {expected_op_idx}, got {op_idx}."
                )
            jobs[current_job].append([machine, duration])

    if not jobs:
        raise ValueError("Failed to parse any jobs from prompt_text.")

    max_job_id = max(jobs.keys())
    if strict and set(jobs.keys()) != set(range(max_job_id + 1)):
        raise ValueError("Job IDs must be contiguous from 0 when strict=True.")

    inst_for_ortools = [jobs[job_id] for job_id in range(max_job_id + 1)]

    if strict:
        if expected_jobs is not None and expected_jobs != len(inst_for_ortools):
            raise ValueError(
                f"Header jobs ({expected_jobs}) != parsed jobs ({len(inst_for_ortools)})."
            )
        if expected_machines is not None:
            for job_id, job_ops in enumerate(inst_for_ortools):
                if len(job_ops) != expected_machines:
                    raise ValueError(
                        f"Job {job_id} has {len(job_ops)} ops, expected {expected_machines}."
                    )

    return inst_for_ortools

def parse_solution_actions(
    solution_text: str, strict: bool = True
) -> Tuple[List[ParsedTeacherAction], Optional[int]]:
    """
    Parse one-shot output into per-step teacher actions.

    Returns:
        actions in decode order, declared makespan from text if present.
    """
    if not isinstance(solution_text, str) or not solution_text.strip():
        raise ValueError("solution_text must be a non-empty string.")

    matches = SOLUTION_OPERATION_PATTERN.findall(solution_text)
    if not matches:
        raise ValueError("No 'Job X Operation Y, MZ' actions found in solution_text.")

    next_expected_op: Dict[int, int] = {}
    actions: List[ParsedTeacherAction] = []
    for job_str, op_str, machine_str in matches:
        job_id = int(job_str)
        op_idx = int(op_str)
        machine_id = int(machine_str)

        expected_op_idx = next_expected_op.get(job_id, 0)
        if strict and op_idx != expected_op_idx:
            raise ValueError(
                f"Teacher action order violation for job {job_id}: "
                f"expected op {expected_op_idx}, got {op_idx}."
            )

        next_expected_op[job_id] = op_idx + 1
        actions.append(ParsedTeacherAction(job_id=job_id, op_idx=op_idx, machine_id=machine_id))

    makespan_match = MAKESPAN_PATTERN.search(solution_text)
    declared_makespan = int(makespan_match.group(1)) if makespan_match else None
    return actions, declared_makespan


def _serial_teacher_job_sequence(
    teacher_actions: Sequence[ParsedTeacherAction],
) -> List[int]:
    return [int(action.job_id) for action in teacher_actions]


def build_dispatch_teacher_actions(
    inst_for_ortools: Sequence[Sequence[Sequence[int]]],
    teacher_actions: Sequence[ParsedTeacherAction],
) -> Tuple[List[ParsedTeacherAction], Dict[str, int]]:
    """
    Convert serial teacher order into a dispatch-valid decision order.

    The raw one-shot teacher order can choose jobs that are not dispatchable at the
    current event time. To create dispatch supervision, we first reconstruct the
    target schedule with the serial environment, then replay the same operations in
    an event-driven dispatch environment using a deterministic tie-break:

    1. Prefer feasible jobs whose target schedule start_time equals current_time.
    2. If none exist, choose the feasible job with the earliest target start_time.

    Step 2 is a best-effort fallback for schedules that are not strictly non-delay.
    """
    serial_env = StaticJSSPStepEnv(inst_for_ortools)
    serial_env.rollout_teacher(_serial_teacher_job_sequence(teacher_actions))
    target_event_by_key: Dict[Tuple[int, int], Dict[str, int]] = {
        (int(event["job_id"]), int(event["op_idx"])): dict(event)
        for event in serial_env.get_event_log()
    }

    dispatch_env = DispatchJSSPStepEnv(inst_for_ortools)
    dispatch_env.reset()

    dispatch_actions: List[ParsedTeacherAction] = []
    exact_decisions = 0
    projected_decisions = 0

    while not dispatch_env.is_done():
        state_json = dispatch_env.get_state_json()
        current_time = int(state_json["current_time"])
        feasible_jobs = list(state_json["feasible_jobs"])
        if not feasible_jobs:
            raise RuntimeError(
                "Dispatch environment reached a non-decision epoch while rebuilding teacher."
            )

        candidates: List[Tuple[int, int, int, int]] = []
        for job_id in feasible_jobs:
            op_idx = int(dispatch_env.job_next_op[job_id])
            target_event = target_event_by_key.get((int(job_id), int(op_idx)))
            if target_event is None:
                raise KeyError(
                    f"Missing target event for dispatch teacher rebuild: job={job_id}, op={op_idx}"
                )
            candidates.append(
                (
                    int(target_event["start_time"]),
                    int(target_event["machine_id"]),
                    int(job_id),
                    int(op_idx),
                )
            )

        exact_candidates = [item for item in candidates if int(item[0]) == current_time]
        if exact_candidates:
            chosen_start, chosen_machine, chosen_job, chosen_op_idx = min(exact_candidates)
            exact_decisions += 1
        else:
            chosen_start, chosen_machine, chosen_job, chosen_op_idx = min(candidates)
            projected_decisions += 1

        dispatch_actions.append(
            ParsedTeacherAction(
                job_id=int(chosen_job),
                op_idx=int(chosen_op_idx),
                machine_id=int(chosen_machine),
            )
        )
        dispatch_env.step(int(chosen_job))

    meta = {
        "dispatch_exact_decisions": int(exact_decisions),
        "dispatch_projected_decisions": int(projected_decisions),
    }
    return dispatch_actions, meta

class StaticJSSPStepEnv:
    """
    Deterministic static JSSP environment for step-by-step action selection.

    Action:
        choose one feasible job id at each step.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.scheduled_ops = 0
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(cls, prompt_text: str, strict: bool = True) -> "StaticJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.scheduled_ops = 0
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def get_feasible_jobs(self) -> List[int]:
        return [
            job_id
            for job_id in range(self.num_jobs)
            if self.job_next_op[job_id] < self.operations_per_job[job_id]
        ]

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines

        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1

        return machine_remaining_load, machine_remaining_ops

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(
                float(total_ops - rem_ops) / float(total_ops)
            )
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")

        op_idx = self.job_next_op[job_id]
        if op_idx >= self.operations_per_job[job_id]:
            raise ValueError(f"Job {job_id} has no remaining operation.")

        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = max(self.job_ready_time[job_id], self.machine_ready_time[machine_id])
        end_time = start_time + duration

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": job_id,
            "op_idx": op_idx,
            "machine_id": machine_id,
            "duration": duration,
            "start_time": start_time,
            "end_time": end_time,
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        """
        Roll out a full teacher sequence and collect step-level records.
        """
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return list(self.event_log)

def _format_machine_ready(machine_ready_time: Sequence[int]) -> str:
    return " ".join(f"M{i}={t}" for i, t in enumerate(machine_ready_time))

def _format_feasible_jobs(feasible_jobs: Sequence[int]) -> str:
    if not feasible_jobs:
        return "[]"
    return "[" + ", ".join(f"Job {j}" for j in feasible_jobs) + "]"

def _format_action_codes(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_route_tokens(route_tokens: Sequence[object]) -> str:
    if not route_tokens:
        return "[]"
    return "[" + ", ".join(str(token) for token in route_tokens) + "]"

def _route_contains_machine(route_tokens: Sequence[object], machine_id: int) -> bool:
    if machine_id < 0:
        return False
    prefix = f"M{machine_id}:"
    return any(str(token).startswith(prefix) for token in route_tokens)

def _effect_bottleneck_relation(
    effect: Dict[str, object],
    bottleneck_machine_id: int,
) -> str:
    machine_id = int(effect.get("machine_id", -1))
    remaining_ops_after = int(effect.get("remaining_ops_after", 0))
    post_route_tokens = effect.get("post_route_tokens", [])
    if bottleneck_machine_id < 0:
        return "unknown"
    if machine_id == bottleneck_machine_id:
        return "direct"
    if remaining_ops_after > 0 and _route_contains_machine(post_route_tokens, bottleneck_machine_id):
        return "releases_future_bottleneck"
    return "indirect"

def _effect_summary_lines(
    effect: Dict[str, object],
    state_json: Optional[Dict[str, object]] = None,
) -> List[str]:
    bottleneck_machine_id = (
        int(state_json.get("bottleneck_machine_id", -1))
        if state_json is not None
        else -1
    )
    return [
        f"- operation machine: {_machine_label(effect['machine_id'])}",
        f"- bottleneck_relation: {_effect_bottleneck_relation(effect, bottleneck_machine_id)}",
        f"- cmax: {int(effect['current_cmax_before'])}->{int(effect['current_cmax_after'])}",
        f"- delta_cmax: {int(effect['delta_cmax'])}",
        f"- est_start: {int(effect['estimated_start'])}",
        f"- est_end: {int(effect['estimated_end'])}",
        f"- machine_idle_gap: {int(effect['machine_idle_gap'])}",
        f"- job_wait: {int(effect['job_wait'])}",
        f"- remaining_ops_after: {int(effect['remaining_ops_after'])}",
        f"- remaining_work_after: {int(effect['remaining_work_after'])}",
        f"- machine_load: {int(effect['affected_machine_load'])}",
        f"- post_route: {_format_route_tokens(effect.get('post_route_tokens', []))}",
    ]

def build_reason_input_text(
    state_text: str,
    chosen_action_code: str,
    chosen_effect: Dict[str, object],
    action_effects: Sequence[Dict[str, object]],
    top_k: int = 3,
    state_json: Optional[Dict[str, object]] = None,
) -> str:
    alternatives = _sorted_reason_alternatives(
        action_effects=action_effects,
        chosen_action_code=chosen_action_code,
        top_k=top_k,
    )
    bottleneck_machine_id = (
        int(state_json.get("bottleneck_machine_id", -1))
        if state_json is not None
        else -1
    )
    bottleneck_load = (
        int(state_json.get("bottleneck_machine_load", 0))
        if state_json is not None
        else 0
    )
    bottleneck_ops_left = (
        int(state_json.get("bottleneck_machine_ops_left", 0))
        if state_json is not None
        else 0
    )
    lines = [
        "You are analyzing an already-selected JSSP action.",
        (
            "Objective: explain why this action was selected and why strong alternatives "
            "were not selected."
        ),
        (
            "Ground the explanation in explicit evidence only: immediate timing/Cmax impact, "
            "bottleneck-machine use or release, post-route exposure, "
            "waiting/idle trade-offs, and contrast against top alternatives."
        ),
        "",
        state_text,
        "",
        f"Selected action: {chosen_action_code}",
        "Critical context:",
        (
            f"- bottleneck_machine: {_machine_label(bottleneck_machine_id)} "
            f"(remaining_load={bottleneck_load}, ops_left={bottleneck_ops_left})"
        ),
        "Chosen transition:",
        *_effect_summary_lines(chosen_effect, state_json=state_json),
        "",
        "Top alternatives:",
    ]
    if alternatives:
        for alt in alternatives:
            lines.append(
                (
                    f"- {alt['action_code']}: "
                    f"operation machine={_machine_label(alt['machine_id'])}, "
                    f"post_route={_format_route_tokens(alt.get('post_route_tokens', []))}, "
                    f"cmax {int(alt['current_cmax_before'])}->{int(alt['current_cmax_after'])}, "
                    f"delta_cmax={int(alt['delta_cmax'])}, "
                    f"est_start={int(alt['estimated_start'])}, "
                    f"est_end={int(alt['estimated_end'])}, "
                    f"job_wait={int(alt['job_wait'])}, "
                    f"machine_idle_gap={int(alt['machine_idle_gap'])}, "
                    f"remaining_work_after={int(alt['remaining_work_after'])}, "
                    f"machine_load={int(alt['affected_machine_load'])}, "
                    f"bottleneck_relation={_effect_bottleneck_relation(alt, bottleneck_machine_id)}"
                )
            )
    else:
        lines.append("- (none)")

    lines.extend(
        [
            "",
            "Output format:",
            "Reason: ...",
            "Not chosen:",
            "- <Axxxx>: ...",
        ]
    )
    return "\n".join(lines)

def _chosen_bottleneck_clause(
    effect: Dict[str, object],
    bottleneck_machine_id: int,
    bottleneck_load: int,
    bottleneck_ops_left: int,
) -> str:
    machine_id = int(effect.get("machine_id", -1))
    affected_machine_load = int(effect.get("affected_machine_load", 0))
    post_route = _format_route_tokens(effect.get("post_route_tokens", []))
    if bottleneck_machine_id >= 0 and machine_id == bottleneck_machine_id:
        return (
            f"It directly activates the current bottleneck machine {_machine_label(machine_id)} "
            f"(remaining_load={bottleneck_load}, ops_left={bottleneck_ops_left}), so delaying this move "
            "would postpone work on the heaviest unresolved machine frontier."
        )
    if bottleneck_machine_id >= 0 and _route_contains_machine(effect.get("post_route_tokens", []), bottleneck_machine_id):
        return (
            f"Its post_route keeps future work on bottleneck "
            f"{_machine_label(bottleneck_machine_id)} visible and ready for earlier activation."
        )
    if bottleneck_machine_id >= 0 and affected_machine_load >= max(1, int(0.75 * bottleneck_load)):
        return (
            f"It works on high-load machine {_machine_label(machine_id)} "
            f"(remaining_load={affected_machine_load}), which is close to the current bottleneck pressure."
        )
    return (
        f"It advances the route on {_machine_label(machine_id)} without adding unnecessary "
        "timing friction to the current schedule state."
    )

def _chosen_progress_clause(effect: Dict[str, object]) -> str:
    rem_ops_before = int(effect.get("remaining_ops_before", 0))
    rem_ops_after = int(effect.get("remaining_ops_after", 0))
    rem_work_before = int(effect.get("remaining_work_before", 0))
    rem_work_after = int(effect.get("remaining_work_after", 0))
    if rem_ops_after <= 0:
        return (
            f"It completes this job, eliminating the remaining work from {rem_work_before} to 0."
        )
    post_route = _format_route_tokens(effect.get("post_route_tokens", []))
    if post_route != "[]":
        return (
            f"It reduces this job from remaining_work {rem_work_before}->{rem_work_after} "
            f"and remaining_ops {rem_ops_before}->{rem_ops_after}, leaving post_route={post_route}."
        )
    return (
        f"It reduces this job from remaining_work {rem_work_before}->{rem_work_after} "
        f"and remaining_ops {rem_ops_before}->{rem_ops_after}."
    )

def _chosen_wait_clause(effect: Dict[str, object]) -> Optional[str]:
    machine_idle_gap = int(effect.get("machine_idle_gap", 0))
    job_wait = int(effect.get("job_wait", 0))
    machine_id = int(effect.get("machine_id", -1))
    if machine_idle_gap > 0:
        return (
            f"It fills machine idle gap={machine_idle_gap} on {_machine_label(machine_id)}, "
            "which helps avoid leaving that machine unused."
        )
    if job_wait > 0:
        return (
            f"It intentionally waits {job_wait} time units for {_machine_label(machine_id)}, "
            "indicating that accessing this machine/route is worth the queue delay."
        )
    return "It can start immediately with no extra machine idle gap and no queue wait."

def _chosen_tradeoff_clause(
    chosen: Dict[str, object],
    best_alternative: Optional[Dict[str, object]],
) -> str:
    chosen_after = int(chosen.get("current_cmax_after", chosen.get("estimated_makespan_after", 0)))
    chosen_delta = int(chosen.get("delta_cmax", 0))
    if best_alternative is None:
        if chosen_delta == 0:
            return "It fits under the current Cmax and does not increase makespan immediately."
        return (
            f"It changes projected Cmax {int(chosen['current_cmax_before'])}->{chosen_after} "
            f"(delta={chosen_delta})."
        )

    best_after = int(
        best_alternative.get("current_cmax_after", best_alternative.get("estimated_makespan_after", 0))
    )
    gap = chosen_after - best_after
    if gap < 0:
        return (
            f"Among the strongest alternatives, it gives the smallest immediate projected Cmax "
            f"({chosen_after} vs {best_after})."
        )
    if gap == 0:
        if chosen_delta == 0:
            return "Its immediate projected Cmax matches the strongest alternatives while staying under the current makespan frontier."
        return (
            f"Its immediate projected Cmax matches the strongest alternative "
            f"({chosen_after}), so the tie is broken by route and bottleneck considerations."
        )
    if gap <= max(3, int(chosen.get("proc_time", 0)) // 2):
        return (
            f"It accepts only a small immediate Cmax penalty (+{gap} vs the strongest alternative) "
            "in exchange for better bottleneck/route progression."
        )
    return (
        f"Although its immediate projected Cmax is worse by +{gap} than the strongest alternative, "
        "the preference comes from stronger bottleneck alignment and downstream progression rather "
        "than short-term Cmax alone."
    )

def _alt_reason_line(
    alt: Dict[str, object],
    chosen: Dict[str, object],
    bottleneck_machine_id: int,
) -> str:
    alt_after = int(alt.get("current_cmax_after", alt.get("estimated_makespan_after", 0)))
    chosen_after = int(chosen.get("current_cmax_after", chosen.get("estimated_makespan_after", 0)))
    alt_machine = int(alt.get("machine_id", -1))
    chosen_machine = int(chosen.get("machine_id", -1))
    alt_post_route_tokens = alt.get("post_route_tokens", [])
    chosen_post_route_tokens = chosen.get("post_route_tokens", [])

    clauses: List[str] = []
    if alt_after > chosen_after:
        clauses.append(
            f"higher immediate projected Cmax ({alt_after} vs {chosen_after})"
        )
    elif alt_after < chosen_after:
        clauses.append(
            f"smaller immediate projected Cmax ({alt_after} vs {chosen_after}), but weaker structural progression"
        )
    else:
        clauses.append(
            f"same immediate projected Cmax ({alt_after})"
        )

    if bottleneck_machine_id >= 0 and chosen_machine == bottleneck_machine_id and alt_machine != bottleneck_machine_id:
        clauses.append(
            f"does not activate bottleneck {_machine_label(bottleneck_machine_id)}"
        )
    elif (
        bottleneck_machine_id >= 0
        and _route_contains_machine(chosen_post_route_tokens, bottleneck_machine_id)
        and not _route_contains_machine(alt_post_route_tokens, bottleneck_machine_id)
    ):
        clauses.append(
            f"does not pull the future bottleneck step on {_machine_label(bottleneck_machine_id)} forward"
        )
    elif int(alt.get("affected_machine_load", 0)) + 1 < int(chosen.get("affected_machine_load", 0)):
        clauses.append(
            f"works on a lighter machine (load={int(alt.get('affected_machine_load', 0))} vs {int(chosen.get('affected_machine_load', 0))})"
        )

    if int(alt.get("machine_idle_gap", 0)) > int(chosen.get("machine_idle_gap", 0)):
        clauses.append(
            f"larger machine idle gap ({int(alt.get('machine_idle_gap', 0))} vs {int(chosen.get('machine_idle_gap', 0))})"
        )
    elif int(alt.get("job_wait", 0)) > int(chosen.get("job_wait", 0)):
        clauses.append(
            f"more waiting before start ({int(alt.get('job_wait', 0))} vs {int(chosen.get('job_wait', 0))})"
        )

    rem_work_after = int(alt.get("remaining_work_after", 0))
    rem_ops_after = int(alt.get("remaining_ops_after", 0))
    post_route = _format_route_tokens(alt_post_route_tokens)
    if rem_ops_after > 0 and post_route != "[]":
        clauses.append(
            f"after this move it still leaves {rem_work_after} work and {rem_ops_after} ops, with post_route={post_route}"
        )
    else:
        clauses.append(
            f"after this move it still leaves {rem_work_after} work and {rem_ops_after} ops on that job"
        )

    clauses = clauses[:4]
    return "; ".join(clauses) + "."

def build_teacher_step_rationale(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    chosen_job: int,
    action_code_to_job: Optional[Dict[str, int]] = None,
    max_not_chosen: int = 6,
    compute_action_effects_fn=None,
) -> str:
    """
    Build compact deterministic rationale text for supervision.

    Output format:
        Reason: ...
        Not chosen:
        - Job k: ...
    """
    feasible = [int(j) for j in feasible_jobs]
    chosen = int(chosen_job)
    if chosen not in feasible:
        feasible = feasible + [chosen]

    label_by_job: Dict[int, str]
    action_code_by_job: Dict[int, str]
    if action_code_to_job:
        label_by_job = invert_action_code_map(action_code_to_job)
        if chosen not in label_by_job:
            raise ValueError(
                f"chosen_job={chosen} is not found in action_code_to_job={action_code_to_job}"
            )
        action_code_by_job = dict(label_by_job)
    else:
        label_by_job = {j: f"Job {j}" for j in feasible}
        action_code_by_job = {j: f"Job {j}" for j in feasible}

    compute_fn = compute_action_effects_fn or compute_action_transition_features
    _, action_effects = compute_fn(
        state_json=state_json,
        action_code_to_job={action_code_by_job[j]: j for j in feasible},
    )
    feats = {int(effect["job_id"]): effect for effect in action_effects}
    c = feats[chosen]
    chosen_label = label_by_job[chosen]
    bottleneck_machine_id = int(state_json.get("bottleneck_machine_id", -1))
    bottleneck_load = int(state_json.get("bottleneck_machine_load", 0))
    bottleneck_ops_left = int(state_json.get("bottleneck_machine_ops_left", 0))

    best_alternative: Optional[Dict[str, object]] = None
    if len(feasible) > 1:
        alt_candidates = [feats[j] for j in feasible if j != chosen and j in feats]
        if alt_candidates:
            alt_candidates.sort(
                key=lambda f: (
                    int(f["estimated_makespan_after"]),
                    int(f["delta_cmax"]),
                    int(f["estimated_start"]),
                    int(f["proc_time"]),
                    int(f["job_id"]),
                )
            )
            best_alternative = alt_candidates[0]

    clauses = [
        (
            f"{chosen_label} is feasible on {_machine_label(c['machine_id'])} "
            f"(est_start={int(c['estimated_start'])}, est_end={int(c['estimated_end'])})."
        ),
        (
            f"Projected Cmax changes {int(c['current_cmax_before'])}->{int(c['current_cmax_after'])} "
            f"(delta={int(c['delta_cmax'])})."
        ),
        _chosen_progress_clause(c),
        _chosen_bottleneck_clause(
            c,
            bottleneck_machine_id=bottleneck_machine_id,
            bottleneck_load=bottleneck_load,
            bottleneck_ops_left=bottleneck_ops_left,
        ),
        _chosen_wait_clause(c),
        _chosen_tradeoff_clause(c, best_alternative),
    ]

    reason_line = "Reason: " + " ".join(clause for clause in clauses if clause)

    candidates = [j for j in feasible if j != chosen]
    candidates = [j for j in candidates if j in feats]
    candidates.sort(
        key=lambda j: (
            int(feats[j]["estimated_makespan_after"]),
            int(feats[j]["estimated_start"]),
            int(feats[j]["proc_time"]),
            j,
        )
    )
    limited = candidates[: max(0, int(max_not_chosen))]

    lines = [reason_line, "Not chosen:"]
    if not limited:
        lines.append("- (none)")
        return "\n".join(lines)

    for j in limited:
        f = feats[j]
        label = label_by_job[j]
        lines.append(f"- {label}: {_alt_reason_line(f, c, bottleneck_machine_id)}")
    return "\n".join(lines)

class DispatchJSSPStepEnv:
    """
    Event-driven JSSP environment.

    At each decision epoch, the agent chooses one dispatchable job.
    If more dispatchable jobs remain at the same current_time, the next decision
    happens without advancing time. If no dispatchable job remains, current_time
    jumps to the next operation completion event.
    """

    def __init__(self, inst_for_ortools: Sequence[Sequence[Sequence[int]]]):
        if not inst_for_ortools:
            raise ValueError("inst_for_ortools must not be empty.")

        self.inst_for_ortools: List[List[Tuple[int, int]]] = []
        for job_ops in inst_for_ortools:
            parsed_job_ops: List[Tuple[int, int]] = []
            for op in job_ops:
                if len(op) != 2:
                    raise ValueError(f"Each operation must be [machine, duration], got {op}.")
                machine, duration = int(op[0]), int(op[1])
                parsed_job_ops.append((machine, duration))
            self.inst_for_ortools.append(parsed_job_ops)

        self.num_jobs = len(self.inst_for_ortools)
        self.operations_per_job = [len(job_ops) for job_ops in self.inst_for_ortools]
        self.job_total_ops = list(self.operations_per_job)
        self.job_total_work = [
            sum(duration for _, duration in job_ops) for job_ops in self.inst_for_ortools
        ]
        self.total_ops = sum(self.operations_per_job)

        if self.total_ops <= 0:
            raise ValueError("total_ops must be positive.")

        max_machine_id = max(
            machine for job_ops in self.inst_for_ortools for machine, _ in job_ops
        )
        self.num_machines = max_machine_id + 1

        self.job_next_op: List[int] = []
        self.job_ready_time: List[int] = []
        self.machine_ready_time: List[int] = []
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops: List[Dict[str, int]] = []
        self.event_log: List[Dict[str, int]] = []
        self.reset()

    @classmethod
    def from_prompt_jobs_first(
        cls, prompt_text: str, strict: bool = True
    ) -> "DispatchJSSPStepEnv":
        inst_for_ortools = parse_prompt_jobs_first(prompt_text, strict=strict)
        return cls(inst_for_ortools)

    def reset(self) -> Dict[str, object]:
        self.job_next_op = [0] * self.num_jobs
        self.job_ready_time = [0] * self.num_jobs
        self.machine_ready_time = [0] * self.num_machines
        self.current_time = 0
        self.scheduled_ops = 0
        self.running_ops = []
        self.event_log = []
        return self.get_state_json()

    def is_done(self) -> bool:
        return self.scheduled_ops == self.total_ops

    def get_makespan(self) -> int:
        return max(self.machine_ready_time) if self.machine_ready_time else 0

    def _next_operation(self, job_id: int, offset: int = 0) -> Tuple[int, int]:
        op_idx = self.job_next_op[job_id] + int(offset)
        if op_idx >= self.operations_per_job[job_id]:
            return -1, 0
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        return int(machine_id), int(duration)

    def _remaining_work(self, job_id: int) -> int:
        next_op = self.job_next_op[job_id]
        return sum(duration for _, duration in self.inst_for_ortools[job_id][next_op:])

    def _post_route_tokens(self, job_id: int) -> List[str]:
        next_op = self.job_next_op[job_id] + 1
        return [
            f"M{int(machine_id)}:{int(duration)}"
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]
        ]

    def _remaining_machine_loads_and_ops(self) -> Tuple[List[int], List[int]]:
        machine_remaining_load = [0] * self.num_machines
        machine_remaining_ops = [0] * self.num_machines
        for job_id in range(self.num_jobs):
            next_op = self.job_next_op[job_id]
            for machine_id, duration in self.inst_for_ortools[job_id][next_op:]:
                machine_remaining_load[int(machine_id)] += int(duration)
                machine_remaining_ops[int(machine_id)] += 1
        return machine_remaining_load, machine_remaining_ops

    def _dispatchable(self, job_id: int) -> bool:
        if self.job_next_op[job_id] >= self.operations_per_job[job_id]:
            return False
        machine_id, duration = self._next_operation(job_id, offset=0)
        if machine_id < 0 or duration <= 0:
            return False
        return (
            int(self.job_ready_time[job_id]) <= int(self.current_time)
            and int(self.machine_ready_time[machine_id]) <= int(self.current_time)
        )

    def get_feasible_jobs(self) -> List[int]:
        return [job_id for job_id in range(self.num_jobs) if self._dispatchable(job_id)]

    def _advance_to_next_event(self) -> None:
        if not self.running_ops:
            return
        next_time = min(int(op["end_time"]) for op in self.running_ops)
        self.current_time = int(next_time)
        self.running_ops = [
            dict(op) for op in self.running_ops if int(op["end_time"]) > int(self.current_time)
        ]

    def _advance_until_decision_epoch(self) -> None:
        while not self.is_done() and not self.get_feasible_jobs():
            if not self.running_ops:
                raise RuntimeError(
                    "Dispatch environment deadlocked: no dispatchable jobs and no running ops."
                )
            self._advance_to_next_event()

    def get_state_json(self) -> Dict[str, object]:
        next_machine: List[int] = []
        next_proc_time: List[int] = []
        next2_machine: List[int] = []
        next2_proc_time: List[int] = []
        remaining_ops: List[int] = []
        remaining_work: List[int] = []
        remaining_work_ratio: List[float] = []
        job_progress_ratio: List[float] = []
        post_route_tokens: List[List[str]] = []

        for job_id in range(self.num_jobs):
            op_idx = self.job_next_op[job_id]
            machine, duration = self._next_operation(job_id, offset=0)
            machine2, duration2 = self._next_operation(job_id, offset=1)
            next_machine.append(machine)
            next_proc_time.append(duration)
            next2_machine.append(machine2)
            next2_proc_time.append(duration2)

            rem_ops = self.operations_per_job[job_id] - op_idx
            rem_work = self._remaining_work(job_id)
            total_work = max(int(self.job_total_work[job_id]), 1)
            total_ops = max(int(self.job_total_ops[job_id]), 1)

            remaining_ops.append(int(rem_ops))
            remaining_work.append(int(rem_work))
            remaining_work_ratio.append(float(rem_work) / float(total_work))
            job_progress_ratio.append(float(total_ops - rem_ops) / float(total_ops))
            post_route_tokens.append(self._post_route_tokens(job_id))

        current_cmax = self.get_makespan()
        total_remaining_work = int(sum(remaining_work))
        unfinished_jobs_count = sum(1 for x in remaining_ops if int(x) > 0)
        unfinished_jobs_ratio = (
            float(unfinished_jobs_count) / float(self.num_jobs) if self.num_jobs else 0.0
        )
        machine_ready_min = min(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_max = max(self.machine_ready_time) if self.machine_ready_time else 0
        machine_ready_mean = (
            float(sum(self.machine_ready_time)) / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_var = (
            sum((float(x) - machine_ready_mean) ** 2 for x in self.machine_ready_time)
            / float(len(self.machine_ready_time))
            if self.machine_ready_time
            else 0.0
        )
        machine_ready_std = float(machine_ready_var ** 0.5)
        machine_remaining_load, machine_remaining_ops = self._remaining_machine_loads_and_ops()
        bottleneck_machine_load = max(machine_remaining_load) if machine_remaining_load else 0
        bottleneck_machine_id = (
            machine_remaining_load.index(bottleneck_machine_load)
            if machine_remaining_load
            else -1
        )
        bottleneck_machine_ops_left = (
            int(machine_remaining_ops[bottleneck_machine_id])
            if bottleneck_machine_id >= 0
            else 0
        )

        state = {
            "step_idx": self.scheduled_ops,
            "total_steps": self.total_ops,
            "scheduled_ratio": (
                float(self.scheduled_ops) / float(self.total_ops) if self.total_ops else 0.0
            ),
            "current_time": int(self.current_time),
            "current_cmax": int(current_cmax),
            "job_next_op": list(self.job_next_op),
            "job_total_ops": list(self.job_total_ops),
            "job_total_work": list(self.job_total_work),
            "job_ready_time": list(self.job_ready_time),
            "machine_ready_time": list(self.machine_ready_time),
            "next_machine": next_machine,
            "next_proc_time": next_proc_time,
            "next2_machine": next2_machine,
            "next2_proc_time": next2_proc_time,
            "remaining_ops": remaining_ops,
            "remaining_work": remaining_work,
            "remaining_work_ratio": remaining_work_ratio,
            "job_progress_ratio": job_progress_ratio,
            "post_route_tokens": post_route_tokens,
            "total_remaining_work": int(total_remaining_work),
            "unfinished_jobs_count": int(unfinished_jobs_count),
            "unfinished_jobs_ratio": float(unfinished_jobs_ratio),
            "machine_ready_min": int(machine_ready_min),
            "machine_ready_mean": float(machine_ready_mean),
            "machine_ready_max": int(machine_ready_max),
            "machine_ready_std": float(machine_ready_std),
            "machine_remaining_load": machine_remaining_load,
            "machine_remaining_ops": machine_remaining_ops,
            "bottleneck_machine_id": int(bottleneck_machine_id),
            "bottleneck_machine_load": int(bottleneck_machine_load),
            "bottleneck_machine_ops_left": int(bottleneck_machine_ops_left),
            "feasible_jobs": self.get_feasible_jobs(),
            "dispatchable_jobs": self.get_feasible_jobs(),
            "idle_machines": [
                int(machine_id)
                for machine_id, ready in enumerate(self.machine_ready_time)
                if int(ready) <= int(self.current_time)
            ],
            "running_operations": [dict(op) for op in sorted(
                self.running_ops,
                key=lambda x: (int(x["end_time"]), int(x["machine_id"]), int(x["job_id"])),
            )],
        }
        return state

    def step(self, job_id: int) -> Tuple[Dict[str, object], float, bool, Dict[str, int]]:
        if self.is_done():
            raise ValueError("Cannot step: environment is already done.")
        if job_id < 0 or job_id >= self.num_jobs:
            raise ValueError(f"Invalid job_id: {job_id}.")
        feasible_jobs = self.get_feasible_jobs()
        if job_id not in feasible_jobs:
            raise ValueError(
                f"Job {job_id} is not dispatchable at t={self.current_time}. feasible={feasible_jobs}"
            )

        op_idx = self.job_next_op[job_id]
        machine_id, duration = self.inst_for_ortools[job_id][op_idx]
        start_time = int(self.current_time)
        end_time = start_time + int(duration)

        self.job_ready_time[job_id] = end_time
        self.machine_ready_time[machine_id] = end_time
        self.job_next_op[job_id] += 1
        self.scheduled_ops += 1
        self.running_ops.append(
            {
                "job_id": int(job_id),
                "op_idx": int(op_idx),
                "machine_id": int(machine_id),
                "start_time": int(start_time),
                "end_time": int(end_time),
                "duration": int(duration),
            }
        )

        event = {
            "step_idx": self.scheduled_ops - 1,
            "job_id": int(job_id),
            "op_idx": int(op_idx),
            "machine_id": int(machine_id),
            "duration": int(duration),
            "start_time": int(start_time),
            "end_time": int(end_time),
            "decision_time": int(start_time),
            "makespan_so_far": self.get_makespan(),
        }
        self.event_log.append(event)

        done = self.is_done()
        if not done:
            self._advance_until_decision_epoch()
        next_state = self.get_state_json()
        reward = 0.0
        return next_state, reward, done, event

    def rollout_teacher(self, job_sequence: Iterable[int]) -> List[Dict[str, object]]:
        self.reset()
        records: List[Dict[str, object]] = []
        for step_idx, job_id in enumerate(job_sequence):
            state_before = self.get_state_json()
            feasible_jobs = state_before["feasible_jobs"]
            if job_id not in feasible_jobs:
                raise ValueError(
                    f"Infeasible teacher action at step {step_idx}: "
                    f"job {job_id}, feasible={feasible_jobs}."
                )
            _, _, done, info = self.step(job_id)
            records.append(
                {
                    "step_idx": step_idx,
                    "state_json": state_before,
                    "feasible_jobs": list(feasible_jobs),
                    "target_job": job_id,
                    "info": info,
                    "done": done,
                }
            )
        if not self.is_done():
            raise ValueError(
                f"Teacher sequence ended early: scheduled {self.scheduled_ops}/{self.total_ops}."
            )
        return records

    def get_event_log(self) -> List[Dict[str, int]]:
        return [dict(event) for event in self.event_log]

def _format_action_codes_dispatch(action_codes: Sequence[str]) -> str:
    if not action_codes:
        return "[]"
    return "[" + ", ".join(str(code) for code in action_codes) + "]"

def _format_value_dispatch(value: object) -> str:
    if isinstance(value, float):
        return f"{value:.4f}".rstrip("0").rstrip(".")
    return str(value)

def _machine_token_dispatch(machine_id: int) -> str:
    return f"M{int(machine_id)}" if int(machine_id) >= 0 else "M-"

def summarize_dispatch_state(state_json: Dict[str, object]) -> Dict[str, object]:
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    return {
        "step_idx": int(state_json["step_idx"]),
        "total_steps": int(state_json["total_steps"]),
        "scheduled_ratio": float(state_json.get("scheduled_ratio", 0.0)),
        "current_time": int(state_json.get("current_time", 0)),
        "current_cmax": current_cmax,
        "unfinished_jobs_count": int(state_json.get("unfinished_jobs_count", 0)),
        "unfinished_jobs_ratio": float(state_json.get("unfinished_jobs_ratio", 0.0)),
        "machine_ready_min": int(state_json.get("machine_ready_min", 0)),
        "machine_ready_mean": float(state_json.get("machine_ready_mean", 0.0)),
        "machine_ready_max": int(state_json.get("machine_ready_max", 0)),
        "machine_ready_std": float(state_json.get("machine_ready_std", 0.0)),
        "bottleneck_machine_id": int(state_json.get("bottleneck_machine_id", -1)),
        "bottleneck_machine_load": int(state_json.get("bottleneck_machine_load", 0)),
        "bottleneck_machine_ops_left": int(state_json.get("bottleneck_machine_ops_left", 0)),
    }

def compute_action_transition_features_dispatch(
    state_json: Dict[str, object],
    action_code_to_job: Dict[str, int],
) -> Tuple[int, List[Dict[str, object]]]:
    job_ready_time: List[int] = state_json["job_ready_time"]  # type: ignore[assignment]
    machine_ready_time: List[int] = state_json["machine_ready_time"]  # type: ignore[assignment]
    next_machine: List[int] = state_json["next_machine"]  # type: ignore[assignment]
    next_proc_time: List[int] = state_json["next_proc_time"]  # type: ignore[assignment]
    next2_machine: List[int] = state_json.get("next2_machine", [-1] * len(next_machine))  # type: ignore[assignment]
    next2_proc_time: List[int] = state_json.get("next2_proc_time", [0] * len(next_machine))  # type: ignore[assignment]
    remaining_ops: List[int] = state_json["remaining_ops"]  # type: ignore[assignment]
    remaining_work: List[int] = state_json["remaining_work"]  # type: ignore[assignment]
    job_total_ops: List[int] = state_json.get("job_total_ops", [1] * len(next_machine))  # type: ignore[assignment]
    job_total_work: List[int] = state_json.get("job_total_work", [1] * len(next_machine))  # type: ignore[assignment]
    machine_remaining_load: List[int] = state_json.get("machine_remaining_load", [0] * len(machine_ready_time))  # type: ignore[assignment]
    machine_remaining_ops: List[int] = state_json.get("machine_remaining_ops", [0] * len(machine_ready_time))  # type: ignore[assignment]
    post_route_tokens_by_job: List[List[str]] = state_json.get("post_route_tokens", [[] for _ in next_machine])  # type: ignore[assignment]

    current_time = int(state_json.get("current_time", 0))
    current_cmax = int(
        state_json.get("current_cmax", max(machine_ready_time) if machine_ready_time else 0)
    )
    total_remaining_work = int(
        state_json.get("total_remaining_work", sum(int(x) for x in remaining_work))
    )

    effects: List[Dict[str, object]] = []
    for action_code, job_id in action_code_to_job.items():
        job = int(job_id)
        machine_id = int(next_machine[job])
        proc_time = int(next_proc_time[job])
        if machine_id < 0 or proc_time <= 0:
            continue

        job_ready_before = int(job_ready_time[job])
        machine_ready_before = int(machine_ready_time[machine_id])
        est_start = max(current_time, job_ready_before, machine_ready_before)
        est_end = est_start + proc_time
        current_cmax_after = max(current_cmax, est_end)
        delta_cmax = current_cmax_after - current_cmax
        rem_ops_before = int(remaining_ops[job])
        rem_ops_after = max(0, rem_ops_before - 1)
        rem_work_before = int(remaining_work[job])
        rem_work_after = max(0, rem_work_before - proc_time)
        total_ops = max(int(job_total_ops[job]), 1)
        total_work = max(int(job_total_work[job]), 1)
        job_progress_ratio_before = float(total_ops - rem_ops_before) / float(total_ops)
        job_progress_ratio_after = float(total_ops - rem_ops_after) / float(total_ops)
        affected_machine_load = int(machine_remaining_load[machine_id])
        affected_machine_ops_left = int(machine_remaining_ops[machine_id])

        effects.append(
            {
                "action_code": str(action_code),
                "job_id": job,
                "machine_id": machine_id,
                "machine_token": _machine_token_dispatch(machine_id),
                "proc_time": proc_time,
                "next_machine": machine_id,
                "next_proc_time": proc_time,
                "next2_machine": int(next2_machine[job]),
                "next2_proc_time": int(next2_proc_time[job]),
                "remaining_ops_before": rem_ops_before,
                "remaining_ops_after": rem_ops_after,
                "remaining_work_before": rem_work_before,
                "remaining_work_after": rem_work_after,
                "job_progress_ratio_before": float(job_progress_ratio_before),
                "job_progress_ratio_after": float(job_progress_ratio_after),
                "job_ready_before": job_ready_before,
                "job_ready_after": int(est_end),
                "machine_ready_before": machine_ready_before,
                "machine_ready_after": int(est_end),
                "estimated_start": int(est_start),
                "estimated_end": int(est_end),
                "est_start": int(est_start),
                "est_end": int(est_end),
                "decision_time": int(current_time),
                "current_cmax_before": int(current_cmax),
                "current_cmax_after": int(current_cmax_after),
                "estimated_makespan_after": int(current_cmax_after),
                "delta_cmax": int(delta_cmax),
                "delta_cmax_ratio": (
                    float(delta_cmax) / float(max(current_cmax_after, 1))
                    if float(current_cmax_after) != 0.0
                    else 0.0
                ),
                "job_wait": int(max(0, current_time - job_ready_before)),
                "machine_idle_gap": int(max(0, current_time - machine_ready_before)),
                "slack_to_current_cmax": int(current_cmax - est_end),
                "affected_machine_load": affected_machine_load,
                "affected_machine_ops_left": affected_machine_ops_left,
                "affected_machine_load_ratio": (
                    float(affected_machine_load) / float(max(total_remaining_work, 1))
                ),
                "remaining_work_after_ratio": float(rem_work_after) / float(total_work),
                "post_route_tokens": list(post_route_tokens_by_job[job]),
                "post_route_len": int(len(post_route_tokens_by_job[job])),
            }
        )

    effects.sort(
        key=lambda x: (
            int(x["estimated_makespan_after"]),
            int(x["estimated_start"]),
            int(x["proc_time"]),
            int(x["job_id"]),
        )
    )
    return int(current_cmax), effects

def render_action_transition_line_dispatch(effect: Dict[str, object]) -> str:
    return (
        f"{effect['action_code']} | "
        f"operation machine={_machine_token(int(effect['next_machine']))} | "
        f"processing time={effect['next_proc_time']} | "
        f"decision_t={effect['decision_time']} | "
        f"est_start={effect['estimated_start']} | "
        f"est_end={effect['estimated_end']} | "
        f"cmax:{effect['current_cmax_before']}->{effect['current_cmax_after']} | "
        f"delta_cmax={effect['delta_cmax']} | "
        f"job_ready:{effect['job_ready_before']}->{effect['job_ready_after']} | "
        f"machine_ready:{effect['machine_ready_before']}->{effect['machine_ready_after']} | "
        f"rem_ops:{effect['remaining_ops_before']}->{effect['remaining_ops_after']} | "
        f"rem_work:{effect['remaining_work_before']}->{effect['remaining_work_after']} | "
        f"machine_load={effect['affected_machine_load']} | "
        f"machine_ops_left={effect['affected_machine_ops_left']} | "
        f"rem_work_after_ratio={_format_value(effect['remaining_work_after_ratio'])} | "
        f"post_route={_format_route_tokens(effect.get('post_route_tokens', []))}"
    )

def _order_prompt_effects_randomly_dispatch(
    state_json: Dict[str, object],
    action_effects: Sequence[Dict[str, object]],
) -> List[Dict[str, object]]:
    ordered_effects = [dict(effect) for effect in action_effects]
    seed_material = (
        f"{int(state_json.get('step_idx', 0))}|"
        f"{int(state_json.get('current_time', 0))}|"
        f"{int(state_json.get('current_cmax', 0))}|"
        + "|".join(sorted(str(effect.get("action_code", "")) for effect in ordered_effects))
    )
    rng = random.Random(seed_material)
    rng.shuffle(ordered_effects)
    return ordered_effects

def build_step_prompt_dispatch(
    state_json: Dict[str, object],
    feasible_jobs: Sequence[int],
    step_idx: int,
    total_steps: int,
    problem_context_text: Optional[str] = None,
    action_code_to_job: Optional[Dict[str, int]] = None,
) -> str:
    lines = [
        "You are solving JSSP with event-driven dispatching.",
        "Objective: minimize final makespan (Cmax) while respecting precedence and machine availability.",
    ]
    if problem_context_text:
        lines.extend(["Static problem context:", problem_context_text])

    summary = summarize_dispatch_state(state_json)
    idle_machines = [int(x) for x in state_json.get("idle_machines", [])]  # type: ignore[arg-type]
    running_operations = list(state_json.get("running_operations", []) or [])
    lines.extend(
        [
            "Dispatch state:",
            (
                f"decision_step={int(summary['step_idx']) + 1}/{int(summary['total_steps'])} "
                f"scheduled_ratio={_format_value_dispatch(summary['scheduled_ratio'])}"
            ),
            f"current_time={summary['current_time']}",
            f"current_cmax={summary['current_cmax']}",
            (
                f"unfinished_jobs_count={summary['unfinished_jobs_count']} "
                f"unfinished_jobs_ratio={_format_value_dispatch(summary['unfinished_jobs_ratio'])}"
            ),
            (
                f"idle_machines={[ _machine_token_dispatch(m) for m in idle_machines ]} "
                f"num_running_ops={len(running_operations)}"
            ),
            (
                f"machine_ready_min={summary['machine_ready_min']} "
                f"machine_ready_mean={_format_value_dispatch(summary['machine_ready_mean'])} "
                f"machine_ready_max={summary['machine_ready_max']} "
                f"machine_ready_std={_format_value_dispatch(summary['machine_ready_std'])}"
            ),
            (
                f"bottleneck_machine={_machine_token_dispatch(int(summary['bottleneck_machine_id']))} "
                f"bottleneck_load={summary['bottleneck_machine_load']} "
                f"bottleneck_ops_left={summary['bottleneck_machine_ops_left']}"
            ),
        ]
    )
    if running_operations:
        lines.append("Running operations:")
        for op in running_operations:
            lines.append(
                f"Job {int(op['job_id'])} Op {int(op['op_idx'])} on "
                f"{_machine_token_dispatch(int(op['machine_id']))}: "
                f"{int(op['start_time'])}->{int(op['end_time'])}"
            )

    if action_code_to_job:
        action_codes = list(action_code_to_job.keys())
        _, action_effects = compute_action_transition_features_dispatch(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        lines.extend(
            [
                f"Dispatchable action codes now: {_format_action_codes_dispatch(action_codes)}",
                "Candidate dispatch effects:",
            ]
        )
        prompt_effects = _order_prompt_effects_randomly_dispatch(state_json, action_effects)
        for effect in prompt_effects:
            lines.append(render_action_transition_line_dispatch(effect))
        lines.extend(
            [
                "Action codes are randomized at each decision epoch. Do not assume persistent code identity.",
                "Choose exactly one dispatchable action code.",
                "Return exactly one code from the dispatchable action set, for example <A3812>.",
            ]
        )
    else:
        lines.extend(
            [
                f"Dispatchable jobs: {list(int(j) for j in feasible_jobs)}",
                "Choose exactly one dispatchable job.",
                "Return exactly one line: Action: Job <id>",
            ]
        )
    return "\n".join(lines)

@dataclass(frozen=True)
class _StepStack:
    env_cls: type
    build_problem_context_text: callable
    build_randomized_action_code_map: callable
    build_step_prompt: callable
    build_step_improvement_prompt: callable
    compute_action_transition_features: callable
    invert_action_code_map: callable


def resolve_step_stack(env_mode: str = 'serial') -> _StepStack:
    normalized = str(env_mode or 'serial').strip().lower()
    if normalized == 'serial':
        return _StepStack(
            env_cls=StaticJSSPStepEnv,
            build_problem_context_text=build_problem_context_text,
            build_randomized_action_code_map=build_randomized_action_code_map,
            build_step_prompt=build_step_prompt,
            build_step_improvement_prompt=build_step_improvement_prompt,
            compute_action_transition_features=compute_action_transition_features,
            invert_action_code_map=invert_action_code_map,
        )
    if normalized == 'dispatch':
        return _StepStack(
            env_cls=DispatchJSSPStepEnv,
            build_problem_context_text=build_problem_context_text,
            build_randomized_action_code_map=build_randomized_action_code_map,
            build_step_prompt=build_step_prompt_dispatch,
            build_step_improvement_prompt=build_step_improvement_prompt,
            compute_action_transition_features=compute_action_transition_features_dispatch,
            invert_action_code_map=invert_action_code_map,
        )
    raise ValueError(f"Unsupported env_mode={env_mode}. Use 'serial' or 'dispatch'.")

def iter_slice(data, start_idx, end_idx, max_instances):
    start = max(0, int(start_idx))
    end = len(data) if end_idx is None else min(int(end_idx), len(data))
    selected = range(start, end)
    if max_instances is not None:
        selected = list(selected)[: int(max_instances)]
    for idx in selected:
        yield idx, data[idx]

def convert_example_to_step_rows(
    example: Dict[str, object],
    source_index: int,
    instance_id: Optional[str] = None,
    strict: bool = True,
    strict_makespan: bool = False,
    action_code_seed: int = 42,
    action_code_width: int = 4,
    action_code_cap: int = 9999,
    dataset_role: str = "both",
    reason_topk: int = 3,
    env_mode: str = "serial",
) -> Tuple[List[Dict[str, object]], Dict[str, object]]:
    """
    Convert one one-shot JSSP sample into `J*M` step rows.
    """
    prompt_jobs_first = str(example.get("prompt_jobs_first", ""))
    solution_text = str(example.get("output", ""))
    num_jobs = int(example.get("num_jobs", 0))
    num_machines = int(example.get("num_machines", 0))

    inst_for_ortools = parse_prompt_jobs_first(prompt_jobs_first, strict=strict)
    teacher_actions, declared_makespan = parse_solution_actions(solution_text, strict=strict)
    step_stack = resolve_step_stack(env_mode)
    env = step_stack.env_cls(inst_for_ortools)

    if num_jobs and env.num_jobs != num_jobs:
        raise ValueError(
            f"num_jobs mismatch at source_index={source_index}: header {num_jobs}, parsed {env.num_jobs}"
        )
    if num_machines and max(env.operations_per_job) != num_machines:
        raise ValueError(
            f"num_machines mismatch at source_index={source_index}: "
            f"header {num_machines}, parsed max ops/job {max(env.operations_per_job)}"
        )

    expected_steps = env.total_ops
    if len(teacher_actions) != expected_steps:
        raise ValueError(
            f"teacher action count mismatch at source_index={source_index}: "
            f"{len(teacher_actions)} vs expected {expected_steps}"
        )

    dispatch_teacher_meta = {
        "dispatch_exact_decisions": 0,
        "dispatch_projected_decisions": 0,
    }
    if str(env_mode) == "dispatch":
        rollout_actions, dispatch_teacher_meta = build_dispatch_teacher_actions(
            inst_for_ortools=inst_for_ortools,
            teacher_actions=teacher_actions,
        )
    else:
        rollout_actions = list(teacher_actions)

    resolved_instance_id = instance_id or f"train_{source_index:06d}"
    rows: List[Dict[str, object]] = []
    problem_context_text = step_stack.build_problem_context_text(inst_for_ortools)
    env.reset()
    if dataset_role not in {"policy", "reason", "both"}:
        raise ValueError(
            f"Unsupported dataset_role={dataset_role}. Use one of: policy, reason, both."
        )

    for step_idx, action in enumerate(rollout_actions):
        state_json = env.get_state_json()
        feasible_jobs = list(state_json["feasible_jobs"])
        target_job = int(action.job_id)
        target_op_idx = int(action.op_idx)
        target_machine = int(action.machine_id)

        if target_job not in feasible_jobs:
            raise ValueError(
                f"infeasible teacher job at source_index={source_index}, step={step_idx}: "
                f"job={target_job}, feasible={feasible_jobs}"
            )

        expected_op_idx = state_json["job_next_op"][target_job]
        expected_machine = state_json["next_machine"][target_job]
        if strict and target_op_idx != expected_op_idx:
            raise ValueError(
                f"teacher op_idx mismatch at source_index={source_index}, step={step_idx}: "
                f"expected {expected_op_idx}, got {target_op_idx}"
            )
        if strict and target_machine != expected_machine:
            raise ValueError(
                f"teacher machine mismatch at source_index={source_index}, step={step_idx}: "
                f"expected M{expected_machine}, got M{target_machine}"
            )

        step_rng = random.Random(
            int(action_code_seed) + int(source_index) * 1_000_003 + int(step_idx)
        )
        action_code_to_job = step_stack.build_randomized_action_code_map(
            feasible_jobs=feasible_jobs,
            rng=step_rng,
            code_width=action_code_width,
            code_start=1,
            code_cap=action_code_cap,
        )
        job_to_action_code = step_stack.invert_action_code_map(action_code_to_job)
        if target_job not in job_to_action_code:
            raise ValueError(
                f"target_job={target_job} not found in action_code_to_job at source_index={source_index}, "
                f"step={step_idx}, mapping={action_code_to_job}"
            )
        target_action_code = job_to_action_code[target_job]
        feasible_action_codes = list(action_code_to_job.keys())
        _, action_effects = step_stack.compute_action_transition_features(
            state_json=state_json,
            action_code_to_job=action_code_to_job,
        )
        effect_by_code = {
            str(effect["action_code"]): dict(effect)
            for effect in action_effects
        }
        chosen_effect = effect_by_code[str(target_action_code)]
        contrast_effects = _sorted_reason_alternatives(
            action_effects=action_effects,
            chosen_action_code=str(target_action_code),
            top_k=int(reason_topk),
        )

        state_text = step_stack.build_step_prompt(
            state_json=state_json,
            feasible_jobs=feasible_jobs,
            step_idx=step_idx,
            total_steps=expected_steps,
            problem_context_text=problem_context_text,
            action_code_to_job=action_code_to_job,
        )
        target_reason_text = build_teacher_step_rationale(
            state_json=state_json,
            feasible_jobs=feasible_jobs,
            chosen_job=target_job,
            action_code_to_job=action_code_to_job,
            compute_action_effects_fn=step_stack.compute_action_transition_features,
        )
        reason_input_text = build_reason_input_text(
            state_text=state_text,
            chosen_action_code=str(target_action_code),
            chosen_effect=chosen_effect,
            action_effects=action_effects,
            top_k=int(reason_topk),
            state_json=state_json,
        )
        target_action_reason_text = f"{target_action_code}\n{target_reason_text}"
        common_row = {
            "instance_id": resolved_instance_id,
            "source_index": source_index,
            "num_jobs": env.num_jobs,
            "num_machines": env.num_machines,
            "total_steps": expected_steps,
            "step_idx": step_idx,
            "state_json": state_json,
            "state_text": state_text,
            "problem_context_text": problem_context_text,
            "feasible_jobs": feasible_jobs,
            "feasible_action_codes": feasible_action_codes,
            "action_code_to_job": action_code_to_job,
            "teacher_operation_idx": target_op_idx,
            "teacher_machine": target_machine,
            "env_mode": str(env_mode),
            "dispatch_teacher_exact": (
                bool(dispatch_teacher_meta["dispatch_projected_decisions"] == 0)
                if str(env_mode) == "dispatch"
                else None
            ),
        }
        policy_feature_schema = (
            "jssp_step_policy_v2_action_token"
            if str(env_mode) == "serial"
            else "jssp_step_policy_dispatch_v1_action_token"
        )
        reason_feature_schema = (
            "jssp_step_reason_v2_action_token"
            if str(env_mode) == "serial"
            else "jssp_step_reason_dispatch_v1_action_token"
        )
        mixed_feature_schema = (
            "jssp_step_v3_transition_action_token"
            if str(env_mode) == "serial"
            else "jssp_step_dispatch_v1_transition_action_token"
        )
        policy_row = {
            **common_row,
            "feature_schema_version": policy_feature_schema,
            "target_job": target_job,
            "target_action_code": target_action_code,
            "target_text": target_action_code,
        }
        reason_row = {
            **common_row,
            "feature_schema_version": reason_feature_schema,
            "selected_job": target_job,
            "selected_action_code": target_action_code,
            "reason_input_text": reason_input_text,
            "reason_target_text": target_reason_text,
            "chosen_transition_features": chosen_effect,
            "contrast_action_codes": [
                str(effect["action_code"]) for effect in contrast_effects
            ],
            "contrast_transition_features": contrast_effects,
            "reason_source": "deterministic_teacher_v1",
        }
        both_row = {
            **policy_row,
            "feature_schema_version": mixed_feature_schema,
            "target_reason_text": target_reason_text,
            "target_action_reason_text": target_action_reason_text,
            "reason_input_text": reason_input_text,
            "reason_target_text": target_reason_text,
            "selected_job": target_job,
            "selected_action_code": target_action_code,
            "chosen_transition_features": chosen_effect,
            "contrast_action_codes": [
                str(effect["action_code"]) for effect in contrast_effects
            ],
            "contrast_transition_features": contrast_effects,
            "reason_source": "deterministic_teacher_v1",
        }
        if dataset_role == "policy":
            rows.append(policy_row)
        elif dataset_role == "reason":
            rows.append(reason_row)
        else:
            rows.append(both_row)

        env.step(target_job)

    if not env.is_done():
        raise ValueError(
            f"rollout did not finish at source_index={source_index}: "
            f"{env.scheduled_ops}/{env.total_ops}"
        )

    computed_makespan = env.get_makespan()
    if strict_makespan and declared_makespan is not None and computed_makespan != declared_makespan:
        raise ValueError(
            f"makespan mismatch at source_index={source_index}: "
            f"declared={declared_makespan}, computed={computed_makespan}"
        )

    meta = {
        "feature_schema_version": (
            (
                "jssp_step_policy_v2_action_token"
                if str(env_mode) == "serial"
                else "jssp_step_policy_dispatch_v1_action_token"
            )
            if dataset_role == "policy"
            else (
                "jssp_step_reason_v2_action_token"
                if str(env_mode) == "serial"
                else "jssp_step_reason_dispatch_v1_action_token"
            )
            if dataset_role == "reason"
            else (
                "jssp_step_v3_transition_action_token"
                if str(env_mode) == "serial"
                else "jssp_step_dispatch_v1_transition_action_token"
            )
        ),
        "dataset_role": dataset_role,
        "env_mode": str(env_mode),
        "instance_id": resolved_instance_id,
        "source_index": source_index,
        "num_jobs": env.num_jobs,
        "num_machines": env.num_machines,
        "steps": expected_steps,
        "declared_makespan": declared_makespan,
        "computed_makespan": computed_makespan,
        "makespan_match": (
            declared_makespan == computed_makespan if declared_makespan is not None else None
        ),
        "dispatch_exact_decisions": int(dispatch_teacher_meta["dispatch_exact_decisions"]),
        "dispatch_projected_decisions": int(dispatch_teacher_meta["dispatch_projected_decisions"]),
    }
    return rows, meta

def generate_step_dataset(data, output_jsonl_path, output_summary_path, cfg):
    output_path = Path(output_jsonl_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    total_instances = 0
    ok_instances = 0
    skipped_instances = 0
    total_rows = 0
    makespan_mismatch = 0
    dispatch_projected_instances = 0
    dispatch_projected_steps = 0
    failures = []

    with output_path.open('w', encoding='utf-8') as out_f:
        for idx, example in iter_slice(data, cfg['start_idx'], cfg['end_idx'], cfg['max_instances']):
            total_instances += 1
            try:
                rows, meta = convert_example_to_step_rows(
                    example=example,
                    source_index=idx,
                    strict=not cfg['no_strict'],
                    strict_makespan=cfg['strict_makespan'],
                    action_code_seed=int(cfg.get('action_code_seed', 42)),
                    action_code_width=int(cfg.get('action_code_width', 4)),
                    action_code_cap=int(cfg.get('action_code_cap', 9999)),
                    dataset_role=str(cfg.get('dataset_role', 'both')),
                    reason_topk=int(cfg.get('reason_topk', 3)),
                    env_mode=str(cfg.get('env_mode', 'serial')),
                )
                for row in rows:
                    out_f.write(json.dumps(row, ensure_ascii=False) + '\n')
                total_rows += len(rows)
                ok_instances += 1
                if meta['makespan_match'] is False:
                    makespan_mismatch += 1
                dispatch_projected_steps += int(meta.get('dispatch_projected_decisions', 0))
                if int(meta.get('dispatch_projected_decisions', 0)) > 0:
                    dispatch_projected_instances += 1
            except Exception as exc:
                skipped_instances += 1
                failures.append({'source_index': idx, 'error': str(exc)})
                if cfg['fail_fast']:
                    raise
            if total_instances % max(1, int(cfg['progress_every'])) == 0:
                print(
                    f"[progress] processed={total_instances} ok={ok_instances} "
                    f"skipped={skipped_instances} rows={total_rows}"
                )

    summary = {
        'input_source': cfg['input_source'],
        'output_path': str(output_path),
        'processed_instances': total_instances,
        'ok_instances': ok_instances,
        'skipped_instances': skipped_instances,
        'total_rows': total_rows,
        'makespan_mismatch_instances': makespan_mismatch,
        'dataset_role': str(cfg.get('dataset_role', 'both')),
        'env_mode': str(cfg.get('env_mode', 'serial')),
    }
    summary_path = Path(output_summary_path)
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    with summary_path.open('w', encoding='utf-8') as f:
        json.dump({'summary': summary, 'failures': failures[:1000]}, f, ensure_ascii=False, indent=2)
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    print(f'Summary saved to: {summary_path}')
    return output_path, summary_path



## 3) 입력 로드 + 생성 실행


In [ ]:

if CFG.get('input'):
    input_path = CFG['input']
elif CFG['input_source'] == 'hf':
    input_path = hf_hub_download(
        repo_id=CFG['input_hf_repo'],
        repo_type='dataset',
        filename=CFG['input_hf_file'],
        token=HF_TOKEN,
    )
elif CFG['input_source'] == 'local':
    input_path = CFG['input_local_path']
else:
    raise ValueError("CFG['input_source'] must be 'hf' or 'local'")

print('input_path:', input_path)

input_path = Path(input_path)
with input_path.open('r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, dict):
    if isinstance(data.get('data'), list):
        data = data['data']
    else:
        raise ValueError('Expected input JSON to be a list or a dict with key `data`.')
if not isinstance(data, list):
    raise ValueError(f'Expected list data, got {type(data).__name__}')

output_path, summary_path = generate_step_dataset(
    data=data,
    output_jsonl_path=CFG['output_jsonl_path'],
    output_summary_path=CFG['output_summary_path'],
    cfg=CFG,
)

print('generated jsonl:', output_path)
print('generated summary:', summary_path)


## 4) (선택) HF 업로드


In [ ]:
if CFG['upload_output_to_hf']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['output_hf_repo'], repo_type='dataset', private=False, exist_ok=True)

    upload_file(
        path_or_fileobj=str(output_path),
        path_in_repo=CFG['output_hf_path_in_repo'],
        repo_id=CFG['output_hf_repo'],
        repo_type='dataset',
        token=HF_TOKEN,
    )

    if CFG.get('upload_summary_to_hf', False):
        upload_file(
            path_or_fileobj=str(summary_path),
            path_in_repo=CFG['summary_hf_path_in_repo'],
            repo_id=CFG['output_hf_repo'],
            repo_type='dataset',
            token=HF_TOKEN,
        )

    files = api.list_repo_files(repo_id=CFG['output_hf_repo'], repo_type='dataset')
    print(f"upload done: {CFG['output_hf_repo']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_output_to_hf]=False, skip upload')
